In [ ]:
# =======================================================================
# PIPELINE DE FINE-TUNING: BITNET B1.58 + LORA (MEDIASUM v2)
# Formato: Speaker: fala | Template: headline 16 palavras
# =======================================================================

# =======================================================================
# CÉLULA 0 — Configurações de memória
# =======================================================================
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_SKIP_ALLOCATOR_WARMUP"] = "1"

In [ ]:
# =======================================================================
# CÉLULA 1 — Instalação de dependências
# =======================================================================
!pip uninstall torchvision -y
!pip install -q -U transformers datasets peft trl accelerate bitsandbytes
!pip install -q rouge-score nltk evaluate
!pip install -q "torchao>=0.16.0"

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 140.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 50.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 91.3 MB/s eta 0:00:00


In [ ]:
# =======================================================================
# CÉLULA 2 — Imports
# =======================================================================
import os
import torch
import pandas as pd
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from tqdm import tqdm
import json

In [ ]:
# =======================================================================
# CÉLULA 3 — Modelo e Tokenizador
# =======================================================================
model_id = "microsoft/bitnet-b1.58-2B-4T-bf16"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map={"": "cuda:0"},
    attn_implementation="sdpa"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/4.83G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/199 [00:00<?, ?B/s]

In [ ]:
# =======================================================================
# CÉLULA 4 — LoRA (idêntico aos outros fine-tunings)
# =======================================================================
config_lora = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config_lora)
model.print_trainable_parameters()

trainable params: 7,987,200 || all params: 2,420,807,680 || trainable%: 0.3299


In [ ]:
# =======================================================================
# CÉLULA 5 — Download e filtragem do dataset
# =======================================================================
from huggingface_hub import hf_hub_download
import zipfile

print("=========================================================")
print("[PASSO 5] Baixando e filtrando o MediaSum v2...")
print("=========================================================")

# Baixa e extrai os splits necessários
for split in ["train", "val"]:
    print(f"Baixando {split}_data.zip...")
    zip_path = hf_hub_download(
        repo_id="ccdv/mediasum",
        filename=f"{split}_data.zip",
        repo_type="dataset"
    )
    extract_path = f"./mediasum_data/{split}"
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)
    print(f"-> Extraído em: {extract_path}")

# Treino: entre 100 e 1700 palavras, até 15.000 instâncias
dados_treino = []
with open("./mediasum_data/train/train_data.txt", "r") as f:
    for linha in f:
        item = json.loads(linha)
        texto = "\n".join([
            f"{speaker}: {fala}"
            for speaker, fala in zip(item["speaker"], item["utt"])
        ])
        resumo = item["summary"]

        if not texto or not resumo:
            continue

        quantidade_palavras = len(texto.split())
        if 100 < quantidade_palavras <= 1700:
            dados_treino.append({"document": texto, "summary": resumo})

        if len(dados_treino) >= 15000:
            break

# Validação: até 1700 palavras, até 800 instâncias
dados_val = []
with open("./mediasum_data/val/val_data.txt", "r") as f:
    for linha in f:
        item = json.loads(linha)
        texto = "\n".join([
            f"{speaker}: {fala}"
            for speaker, fala in zip(item["speaker"], item["utt"])
        ])
        resumo = item["summary"]

        if not texto or not resumo:
            continue

        if len(texto.split()) <= 1700:
            dados_val.append({"document": texto, "summary": resumo})

        if len(dados_val) >= 800:
            break

if len(dados_treino) == 0:
    raise ValueError("ERRO: filtro eliminou todos os dados de treino.")
if len(dados_val) == 0:
    raise ValueError("ERRO: filtro eliminou todos os dados de validação.")

dataset_treino = Dataset.from_list(dados_treino)
dataset_val = Dataset.from_list(dados_val)

print(f"-> Treino: {len(dataset_treino)} instâncias")
print(f"-> Validação: {len(dataset_val)} instâncias")

[PASSO 5] Baixando e filtrando o MediaSum v2...
Baixando train_data.zip...


train_data.zip:   0%|          | 0.00/1.44G [00:00<?, ?B/s]

-> Extraído em: ./mediasum_data/train
Baixando val_data.zip...


val_data.zip:   0%|          | 0.00/34.1M [00:00<?, ?B/s]

-> Extraído em: ./mediasum_data/val
-> Treino: 15000 instâncias
-> Validação: 800 instâncias


In [ ]:
# =======================================================================
# CÉLULA 6 — Formatação dos prompts
# =======================================================================
def formatar_prompt(exemplo):
    return {
        "text": (
            "### Instruction:\nWrite a short news headline summarizing the main topic of this interview in 16 words or less.\n\n"
            f"### Interview Transcript:\n{exemplo['document']}\n\n"
            f"### Headline:\n{exemplo['summary']}{tokenizer.eos_token}"
        )
    }

dataset_treino_fmt = dataset_treino.map(formatar_prompt)
dataset_val_fmt = dataset_val.map(formatar_prompt)

print("Amostra do prompt formatado:")
print(dataset_treino_fmt[0]["text"][:500])

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Amostra do prompt formatado:
### Instruction:
Write a short news headline summarizing the main topic of this interview in 16 words or less.

### Interview Transcript:
FARAI CHIDEYA, host: Now, moving on, Forest Whitaker as Moses, Tisha Campbell Martin as Mary Magdalene - well, that's all in "The Bible Experience." A New Testament edition was released in 2006. This edition is billed as "The Complete Bible." It doesn't have one person reading the gospels. It features nearly 400 African-American artists, actors and ministers, 


In [ ]:
# =======================================================================
# CÉLULA 7 — Treinamento
# =======================================================================
print("=========================================================")
print("Iniciando Fine-Tuning MediaSum v2...")
print("=========================================================")

args_treino = SFTConfig(
    output_dir="./resultados_mediasum_v2",
    max_length=2048,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    eval_strategy="steps",
    eval_steps=90,
    logging_steps=20,
    learning_rate=2e-4,
    weight_decay=0.01,
    num_train_epochs=1,
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    save_strategy="steps",
    save_steps=200,
    save_total_limit=1,
    report_to="none",
    dataset_text_field="text"
)

trainer = SFTTrainer(
    model=model,
    args=args_treino,
    train_dataset=dataset_treino_fmt,
    eval_dataset=dataset_val_fmt,
)

trainer.train()

Iniciando Fine-Tuning MediaSum v2...


Adding EOS to train dataset:   0%|          | 0/15000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/15000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
90,2.176186,2.428156,2.408839,810717.000000,0.476911
180,2.075289,2.392052,2.340695,1626902.000000,0.483591
270,2.101835,2.389837,2.345337,2439968.000000,0.483821
360,2.071687,2.388870,2.341998,3252547.000000,0.483773
450,2.044374,2.389541,2.334540,4072932.000000,0.483599
540,2.044297,2.388653,2.319251,4881527.000000,0.483549
630,2.053040,2.389402,2.335924,5687081.000000,0.483490
720,2.062870,2.389601,2.329834,6486521.000000,0.483413
810,2.028772,2.387371,2.318771,7302505.000000,0.483601
900,2.054818,2.388282,2.317056,8122017.000000,0.483695


TrainOutput(global_step=1875, training_loss=2.0616547190348307, metrics={'train_runtime': 12185.0821, 'train_samples_per_second': 1.231, 'train_steps_per_second': 0.154, 'total_flos': 2.1261490717114368e+17, 'train_loss': 2.0616547190348307, 'epoch': 1.0})

In [ ]:
# =======================================================================
# CÉLULA 8 — Salvar modelo, histórico e gráficos
# =======================================================================
pasta_salvamento = "./modelo_final_mediasum_v2"
os.makedirs(pasta_salvamento, exist_ok=True)
trainer.model.save_pretrained(pasta_salvamento)
print(f"Pesos LoRA salvos em: {pasta_salvamento}")

history = pd.DataFrame(trainer.state.log_history)
history.to_csv(f"{pasta_salvamento}/historico_treinamento.csv", index=False)
history.to_json(f"{pasta_salvamento}/historico_treinamento.json", orient="records", indent=4)

train_df = history[history['loss'].notna()]
eval_df = history[history['eval_loss'].notna()]

plt.figure(figsize=(10, 5))
plt.plot(train_df['step'], train_df['loss'], label='Train Loss', color='tab:blue')
if not eval_df.empty:
    plt.plot(eval_df['step'], eval_df['eval_loss'], label='Eval Loss', color='tab:orange')

plt.title('BitNet b1.58 + LoRA (MediaSum v2) - Curva de Aprendizado')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.savefig(f"{pasta_salvamento}/loss_curve.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{pasta_salvamento}/loss_curve.pdf", bbox_inches='tight')
plt.close()
print("Gráficos e histórico salvos com sucesso!")

Pesos LoRA salvos em: ./modelo_final_mediasum_v2
Gráficos e histórico salvos com sucesso!


In [ ]:
# =======================================================================
# CÉLULA 9 — Backup e download
# =======================================================================
import shutil
from google.colab import files

pasta_backup = "./backup_mediasum_v2_completo"
os.makedirs(pasta_backup, exist_ok=True)

shutil.copytree("./resultados_mediasum_v2", f"{pasta_backup}/resultados_mediasum_v2", dirs_exist_ok=True)
shutil.copytree("./modelo_final_mediasum_v2", f"{pasta_backup}/modelo_final_mediasum_v2", dirs_exist_ok=True)

shutil.make_archive("./backup_mediasum_v2_completo", "zip", pasta_backup)
print(f"Backup gerado: backup_mediasum_v2_completo.zip ({os.path.getsize('./backup_mediasum_v2_completo.zip') / (1024**2):.1f} MB)")
files.download("./backup_mediasum_v2_completo.zip")

Backup gerado: backup_mediasum_v2_completo.zip (72.9 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>